# LAB 06 – Ciąg Fibonacciego: rekurencja, memoizacja, Bottom-Up

## Zadanie 1 – Rekurencyjna implementacja fib(n)

In [2]:
import time

def fib(n):
    """Rekurencyjna implementacja n-tego wyrazu ciągu Fibonacciego."""
    if n == 1 or n == 2:
        return 1
    return fib(n - 1) + fib(n - 2)

# Scenariusze: a) n=1, b) n=2, c) n=5, d) n=35
# n=100 pominięte – czas wykonania rzędu 2^100 wywołań (praktycznie niewykonalne)
test_cases = [1, 2, 5, 35]
print(f"{'n':>5} | {'fib(n)':>15} | {'czas [s]':>12}")
print("-" * 40)
for n in test_cases:
    t0 = time.perf_counter()
    wynik = fib(n)
    dt = time.perf_counter() - t0
    print(f"{n:>5} | {wynik:>15} | {dt:>12.6f}")

print()
print("Uwaga: fib(100) rekurencyjnie bez memoizacji – czas wykładniczy, niewykonalne w praktyce.")

    n |          fib(n) |     czas [s]
----------------------------------------
    1 |               1 |     0.000003
    2 |               1 |     0.000003
    5 |               5 |     0.000009
   35 |         9227465 |     1.026636

Uwaga: fib(100) rekurencyjnie bez memoizacji – czas wykładniczy, niewykonalne w praktyce.


## Zadanie 2 – Rekurencja z memoizacją

In [3]:
def fib_memo(n, memo=None):
    """Rekurencyjna implementacja fib(n) z memoizacją (cache wyników)."""
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n == 1 or n == 2:
        return 1
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

# Scenariusze a-e – teraz n=100 jest wykonalne w ułamku sekundy
test_cases = [1, 2, 5, 35, 100]
print(f"{'n':>5} | {'fib_memo(n)':>30} | {'czas [s]':>14}")
print("-" * 58)
for n in test_cases:
    t0 = time.perf_counter()
    wynik = fib_memo(n)
    dt = time.perf_counter() - t0
    print(f"{n:>5} | {wynik:>30} | {dt:>14.8f}")

# Porównanie czasu dla n=35: rekurencja vs memoizacja
print()
t0 = time.perf_counter(); _ = fib(35);      dt_rec  = time.perf_counter() - t0
t0 = time.perf_counter(); _ = fib_memo(35); dt_memo = time.perf_counter() - t0

print(f"Porównanie dla n=35:")
print(f"  Rekurencja : {dt_rec:.6f} s")
print(f"  Memoizacja : {dt_memo:.8f} s")
if dt_memo > 0:
    print(f"  Przyspieszenie: ~{dt_rec / dt_memo:.0f}x")

    n |                    fib_memo(n) |       czas [s]
----------------------------------------------------------
    1 |                              1 |     0.00000350
    2 |                              1 |     0.00000270
    5 |                              5 |     0.00000920
   35 |                        9227465 |     0.00001870
  100 |          354224848179261915075 |     0.00004380

Porównanie dla n=35:
  Rekurencja : 1.009886 s
  Memoizacja : 0.00006320 s
  Przyspieszenie: ~15979x


## Zadanie 3 – Podejście Bottom-Up (programowanie dynamiczne)

In [4]:
def fib_bottom_up(n):
    """Bottom-Up DP: budujemy tablicę wyników od podstawy, bez rekurencji. Pamięć O(n)."""
    if n == 1 or n == 2:
        return 1
    dp = [0] * (n + 1)
    dp[1] = 1
    dp[2] = 1
    for i in range(3, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


def fib_bottom_up_opt(n):
    """Bottom-Up z optymalizacją pamięci do O(1) – przechowujemy tylko dwie ostatnie wartości."""
    if n == 1 or n == 2:
        return 1
    prev2, prev1 = 1, 1
    for _ in range(3, n + 1):
        prev2, prev1 = prev1, prev2 + prev1
    return prev1


# Scenariusze a-e
test_cases = [1, 2, 5, 35, 100]
print(f"{'n':>5} | {'fib_bottom_up(n)':>30} | {'czas [s]':>14}")
print("-" * 58)
for n in test_cases:
    t0 = time.perf_counter()
    wynik = fib_bottom_up(n)
    dt = time.perf_counter() - t0
    print(f"{n:>5} | {wynik:>30} | {dt:>14.8f}")

print()
print("Wersja z optymalizacją pamięci O(1):")
for n in test_cases:
    print(f"  fib_bottom_up_opt({n:>3}) = {fib_bottom_up_opt(n)}")

# Zbiorczy benchmark wszystkich metod dla n=35
print()
print("=== Zbiorczy benchmark dla n=35 ===")
N = 35
methods = [
    ("Rekurencja",       lambda: fib(N)),
    ("Memoizacja",       lambda: fib_memo(N)),
    ("Bottom-Up O(n)",   lambda: fib_bottom_up(N)),
    ("Bottom-Up O(1)",   lambda: fib_bottom_up_opt(N)),
]
print(f"{'Metoda':<22} | {'Wynik':>12} | {'Czas [s]':>14}")
print("-" * 54)
for name, fn in methods:
    t0 = time.perf_counter()
    res = fn()
    dt = time.perf_counter() - t0
    print(f"{name:<22} | {res:>12} | {dt:>14.8f}")

    n |               fib_bottom_up(n) |       czas [s]
----------------------------------------------------------
    1 |                              1 |     0.00000190
    2 |                              1 |     0.00000110
    5 |                              5 |     0.00000530
   35 |                        9227465 |     0.00000470
  100 |          354224848179261915075 |     0.00000860

Wersja z optymalizacją pamięci O(1):
  fib_bottom_up_opt(  1) = 1
  fib_bottom_up_opt(  2) = 1
  fib_bottom_up_opt(  5) = 5
  fib_bottom_up_opt( 35) = 9227465
  fib_bottom_up_opt(100) = 354224848179261915075

=== Zbiorczy benchmark dla n=35 ===
Metoda                 |        Wynik |       Czas [s]
------------------------------------------------------
Rekurencja             |      9227465 |     0.96044110
Memoizacja             |      9227465 |     0.00001880
Bottom-Up O(n)         |      9227465 |     0.00000870
Bottom-Up O(1)         |      9227465 |     0.00000350
